## Prática para Entrega: Modelo de Hopfield e BAM
Implementação e teste da rede BAM



1. Crie uma matriz com a representação -1/1 dos dígitos 0, 1, 2, 3, 4, 5, 6, 7, 8 e 9 do conjunto de dados MNIST

2. Crie uma matriz na qual as linhas representam os vetores de saída com a representação -1/1 para cada dígito

3. Execute a rede BAM nesses dados e verifique se os dígitos originais são recuperados corretamente
4. Faça variações nos vetores de dígitos originais, execute a rede BAM, e verifique se os dígitos são recuperados corretamente
5. Compare com a rede de Hopfield vista em aula



In [195]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

Primeiro requesito: Carregar a base de dados - dataset MNIST

In [196]:
def carregando_MNIST(num_desejados):
    mnist =  fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X, y = mnist.data, mnist.target.astype(int)

    padroes_treino = []

    for elem in num_desejados:
        indice = np.where(y == elem)[0][0] #pega a 1° ocoreencia de cada num desejado
        #converte a img -> pixels > 127 vira 1, e restante -1
        img_formatada = np.where(X[indice] > 127, 1, -1)
        padroes_treino.append(img_formatada)

    return np.array(padroes_treino)


In [197]:
padroes = carregando_MNIST([0,1,2,3,4,5,6,7,8,9])


Segundo requisito: matriz na qual as linhas representam os vetores de saída com a representação -1/1 para cada dígito

In [201]:
def matriz_saida( num_desejado):
    #teremos uma matriz num,10
    num = num_desejado.shape[0]
    matriz_saida = -np.ones((num,10))
   

    for i in range(num):
        idx =  (num_desejado[i])
        matriz_saida[i,idx] = 1
    return matriz_saida



In [202]:
print(matriz_saida(np.array([1,3,9,2])))

[[-1.  1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1.  1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1.  1.]
 [-1. -1.  1. -1. -1. -1. -1. -1. -1. -1.]]


implementando a rede BAM

In [203]:
def funcao_ativacao(v):
    #se o elem de v for < 0, troca por -1. caso contrario, +1
    result =  np.where(v < 0, -1, 1)

    #caso seja 0, sorteio aleatorio dos elem -1,1
    zeros = (v ==0)
    result[zeros] = np.random.choice([-1,1], size=zeros.sum())
    return result

In [214]:
def bam_armazenamento(X,y):
    #dimensoes de x e y
    nlinX, ncolX =  X.shape
    _, ncoly =  y.shape
    
    pesos = np.zeros((ncolX, ncoly))

    for i in range(nlinX):
        pesos = pesos + np.outer(X[i,:] ,y[i,])
        
    
    return(pesos)

In [228]:
def bam_teste(M, x):
    entropia_antigo = 100
    delta_entrop = 1
    
    while(delta_entrop > 0):
        #em direcao a y
        y = funcao_ativacao(x @ M)

        #calculo da entropia - valor menor, menos confusao - melhor
        entropia = - (x @ M @ y)

        #em direcao a x
        x_novo = funcao_ativacao(y @ M.T)

        #varicao da entropia
        delta_entrop = abs(entropia - entropia_antigo)
        entropia_antigo = entropia

        print("\nVaricao na Entropia =  ", delta_entrop)

        x = x_novo

    print("Saída: ", y)
    print("Entrada: ", x)


Carregando os numeros

In [229]:
numeros  = np.array([0,1,2,3,4,5,6,7,8,9])
padroes = carregando_MNIST(numeros)
saida =  matriz_saida(numeros)

In [230]:
print(padroes.shape)

(10, 784)


In [231]:

print(saida)

[[ 1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1.  1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1.  1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1.  1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1.  1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1.  1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1.  1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1.  1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1.  1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1.  1.]]


In [232]:
M = bam_armazenamento(padroes, saida)


In [235]:
for i in range(len(padroes)//5):
    bam_teste(M, padroes[i])



Varicao na Entropia =   42628.0

Varicao na Entropia =   5856.0

Varicao na Entropia =   0.0
Saída:  [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
Entrada:  [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  1  1 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  1  1  1  1  1  1  1
  1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  1  1  1  1
  1  1  1  1  1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  1
  1  1 -1 -1 -1 -1  1  1  1 -1 -1 -1 -1 -1 -1 -1 -1 -